# 무료체험 → 결제 전환 EDA

**Business Question**: 어떤 체험 유저가 결제로 전환될 가능성이 높은가?

**네이밍 규칙**: `raw_*` = 원본(전처리 전) · 접두어 없는 이름 = 전처리 후

## 목차
0. [개발환경 세팅](#sec0)
1. [데이터 전처리](#sec1)
    - 1.1 [trial_signup](#sec1-1)  /  1.2 [trial_conversion](#sec1-2)  /  1.3 [access_events](#sec1-3)
    - 1.4 [visit_daily](#sec1-4)  /  1.5 [site_dim](#sec1-5)
    - 1.6 [무료체험 3일 윈도우 정합성 처리](#sec1-6)
    - 1.7 [이상치 처리(stay_time_second)](#sec1-7)
2. [파생변수 생성 & 통합 테이블](#sec2)
3. [EDA](#sec3)
    - 3.1 [방문 여부별](#sec3-1)  /  3.2 [방문일수별](#sec3-2)  /  3.3 [체류시간](#sec3-3)  /  3.4 [지점별](#sec3-4)
    - 3.5 [오전 방문](#sec3-5)  /  3.6 [출입 횟수](#sec3-6)  /  3.7 [방문일수×출입](#sec3-7)
4. [결론](#sec4)


<a id="sec0"></a>
# 0. 개발환경 세팅

In [ ]:
# 패키지
!pip install koreanize_matplotlib -q
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── 한글 폰트: 어느 환경(Colab/Windows/Mac/Linux)에서도 안 깨지게 ──
try:
    import koreanize_matplotlib   # Colab/Linux: 나눔폰트 자동 등록
except Exception:
    pass

_avail = {f.name for f in fm.fontManager.ttflist}
for _f in ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic',
           'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans KR']:
    if _f in _avail:
        plt.rcParams['font.family'] = _f
        break
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지
print('적용 폰트:', plt.rcParams['font.family'])

In [ ]:
# from google.colab import drive   # Colab 전용
# drive.mount('/content/drive/')

## 0.1 공통 경로(PATH) 설정

In [ ]:
PATH = "data/"

<a id="sec1"></a>
# 1. 데이터 전처리

> **네이밍 규칙** — `raw_*` = CSV 원본(전처리 전), 접두어 없는 이름(`register`,`payment`,`access`,`visit`,`site`) = 중복/형변환 등 **전처리 후** 테이블.

<a id="sec1-1"></a>
## 1.1 trial_signup

유저 가입(무료체험 등록) 테이블. `user_uuid` 기준 중복 제거(`keep='last'` = 마지막 체험만 유지) → `register`.

In [ ]:
raw_register = pd.read_csv(PATH + "trial_signup.csv")

In [ ]:
raw_register.head()

In [ ]:
raw_register.info()

In [ ]:
raw_register['trial_date'] = pd.to_datetime(raw_register['trial_date'])
raw_register.dtypes

In [ ]:
raw_register['user_uuid'].duplicated().sum()

In [ ]:
register = raw_register.drop_duplicates(subset=['user_uuid'], keep='last')
register.info()

<a id="sec1-2"></a>
## 1.2 trial_conversion

결제 여부(`is_payment`, 타겟) 테이블. `user_uuid` 기준 중복 제거 → `payment`.

In [ ]:
raw_payment = pd.read_csv(PATH + "trial_conversion.csv")

In [ ]:
raw_payment.head()

In [ ]:
raw_payment.info()

In [ ]:
raw_payment['user_uuid'].duplicated().sum()

In [ ]:
payment = raw_payment.drop_duplicates(subset=['user_uuid'], keep='last')
payment.info()

In [ ]:
payment['is_payment'].value_counts()

In [ ]:
payment['is_payment'].value_counts().plot.pie(autopct='%.1f%%')

<a id="sec1-3"></a>
## 1.3 access_events

출입 로그(1유저 : N행). 완전 중복행 제거 + `cdate` → datetime → `access`.

- 완전 중복행 35건 제거
- `cdate` : object → datetime
- `user_uuid` 중복은 1:N(여러 번 출입) 구조상 정상 → 제거 대상 아님

In [ ]:
# 다른 테이블은 중복 처리 완료
# 제거된 대상의 출입 기록 관리 필요 (마지막 무료체험만 남아있음)

In [ ]:
raw_access = pd.read_csv(PATH + "access_events.csv")

In [ ]:
raw_access.head()

In [ ]:
raw_access.info()

In [ ]:
raw_access.dtypes

In [ ]:
raw_access['cdate'] = pd.to_datetime(raw_access['cdate'])
raw_access.dtypes

In [ ]:
# 한 유저가 들락날락 하기 때문에 user_uuid가 57,682건 잡히는 건 정상적인 구조
# 유저 ID가 아닌 'id' 로 확인해야 함
# user_uuid 중복 카운트는 "유저당 1행이어야 하는 테이블"에서만 의미가 있고, 이벤트/방문 로그에서는 고유키(id) 기준으로 봐야 함
raw_access.duplicated().sum()

In [ ]:
# 완전 중복행(같은 출입 로그 2번) 제거
access = raw_access.drop_duplicates()

In [ ]:
access.info()

In [ ]:
# 유저당 평균 로그 수
access.groupby('user_uuid').size().mean()

In [ ]:
# 유저별 로그 수 중앙값
access.groupby('user_uuid').size().median()

In [ ]:
# 지점별 유저 수
access.groupby('site_id')['user_uuid'].nunique().plot.bar(rot=0)

In [ ]:
# 유저별 빈도수
access.groupby('user_uuid').size().plot.hist(bins=30)

In [ ]:
# 600명정도 입실은 있는데 퇴실은 없음
access['checkin'].value_counts()

### 입실(checkin=1) 유저 기준 분포

In [ ]:
# 입실한 유저 기준
df_checkin_1 = access[access['checkin'] == 1]

In [ ]:
# 유저별 로그 수 중앙값
df_checkin_1.groupby('user_uuid').size().median()

In [ ]:
# 지점별 유저 수
df_checkin_1.groupby('site_id')['user_uuid'].nunique().plot.bar(rot=0)

In [ ]:
# 유저별 빈도수
df_checkin_1.groupby('user_uuid').size().plot.hist(bins=30)

<a id="sec1-4"></a>
## 1.4 visit_daily

방문 단위 요약(1유저 : N행). 완전 중복행만 제거(재입실 세션은 유지) → `visit`.

In [ ]:
raw_visit = pd.read_csv(PATH + "visit_daily.csv")

In [ ]:
raw_visit

### 증복 사례 분류
- 같은 날 재입실 / 실제 데이터 중복 건을 구분해야 함

In [ ]:
cols = ['user_uuid','date','first_enter_time','last_leave_time','stay_time_second']

is_same_ud  = raw_visit.duplicated(['user_uuid','date'], keep=False)  # 같은 user+date 2건 이상
is_full_dup = raw_visit.duplicated(keep=False)                        # 전체 컬럼 완전복제

case_A = raw_visit[is_same_ud & ~is_full_dup]    # A: 같은 user+date지만 완전복제 아님 = 별도 세션
case_B = raw_visit[is_full_dup]                  # B: 완전복제 (같은 기록이 통째로 두 번)

In [ ]:
case_A.sort_values(['user_uuid','date'])[cols]

- 같은 날 시간대가 겹치지 않는 세션이 2건 이상 = 중간 외출 후 재입실 → 별도 방문으로 유지

In [ ]:
case_B.sort_values(['user_uuid','date'])[cols]

- 모든 컬럼이 완전히 같은 행 = 실제 중복 기록 → 제거
- 밤에 입실해 다음 날 새벽에 퇴실한 경우, 자정을 넘긴 연속 체류가 날짜별로 쪼개져 기록됨 → 중복이 아닌 정상 구조로 판단

In [ ]:
# case B(완전복제) 48행만 제거됨 — case A(같은 날 재입실)는 유지
visit = raw_visit.drop_duplicates()

In [ ]:
visit.isnull().sum()

In [ ]:
# format='mixed' 옵션을 주면 소수점이 있는 형태와 없는 형태를 모두 알아서 인식
visit['first_enter_time'] = pd.to_datetime(visit['first_enter_time'], format='mixed')
visit['last_leave_time'] = pd.to_datetime(visit['last_leave_time'], format='mixed')

In [ ]:
visit['stay_time'] = pd.to_timedelta(visit['stay_time'])

In [ ]:
visit.dtypes

<a id="sec1-5"></a>
## 1.5 site_dim

지점(`site_id`)별 면적(평) 마스터. 전처리 불필요 → `site`.

- 별도 특이사항 없음

In [ ]:
raw_site = pd.read_csv(PATH + "site_dim.csv")
site = raw_site.copy()   # 별도 특이사항 없음 → 그대로 사용

In [ ]:
site.info()

<a id="sec1-6"></a>
## 1.6 무료체험 3일 윈도우 정합성 처리 (visit · access)

**문제** — `register`/`payment`는 마지막 체험 1건만 남겼지만 `access`/`visit`에는 같은 유저의 **예전 체험 기록**이 남아 `visit_days`가 3일을 넘어 **4 이상**으로 부풀려짐.

**처리** — 기준일 `D = trial_date`(자정) 기준 윈도우 **D, D+1, D+2(3일)** 밖의 행 제외. 이렇게 하면 2장의 `visit_days = ('date','nunique')`가 자동으로 **0~3** 범위가 된다.

In [ ]:
# 무료체험 3일 윈도우(D ~ D+2)로 visit / access 정합성 처리
trial_ref = register[['user_uuid', 'trial_date']].copy()
trial_ref['trial_day'] = trial_ref['trial_date'].dt.normalize()   # 시작일 자정(D)
before_v, before_a = len(visit), len(access)

# visit: date 가 윈도우 안인 행만
visit['date'] = pd.to_datetime(visit['date'])
visit = visit.merge(trial_ref[['user_uuid', 'trial_day']], on='user_uuid', how='left')
gap_v = (visit['date'].dt.normalize() - visit['trial_day']).dt.days
visit = visit[(gap_v >= 0) & (gap_v <= 2)].drop(columns='trial_day').reset_index(drop=True)

# access: cdate 가 동일 윈도우 안인 행만
access = access.merge(trial_ref[['user_uuid', 'trial_day']], on='user_uuid', how='left')
gap_a = (access['cdate'].dt.normalize() - access['trial_day']).dt.days
access = access[(gap_a >= 0) & (gap_a <= 2)].drop(columns='trial_day').reset_index(drop=True)

print(f'visit  행: {before_v} -> {len(visit)}')
print(f'access 행: {before_a} -> {len(access)}')

In [ ]:
# 검증: 윈도우 적용 후 방문일수 분포 (0~3 이면 정상)
_vd = visit.groupby('user_uuid')['date'].nunique()
print(_vd.value_counts().sort_index())
print('최대 방문일수:', _vd.max())

<a id="sec2"></a>
# 2. 파생변수 생성 & 통합 테이블

분석 단위는 **유저(`user_uuid`) 1명 = 1행**. 1:N 테이블(visit/access)을 `groupby`로 집계해 `register` 기준 **left join**한다. no-show 유저(미방문)가 사라지지 않도록 반드시 left join을 쓰고, 방문 기반 결측은 0/플래그로 채운다.

> `visit_days`는 1.6에서 무료체험 3일 윈도우로 잘라냈으므로 자동으로 **0~3** 범위가 된다.

In [ ]:
# ── 1) visit_daily 집계 (방문 기반 파생변수) ──
g_v = visit.groupby('user_uuid') # 방문 기록 유저별로 묶기

visit_agg = g_v.agg(
    stay_total_sec  = ('stay_time_second', 'sum'),    # 총 체류시간(초)
    stay_mean_sec   = ('stay_time_second', 'mean'),   # 평균 체류시간(초) : 한 번 방문했을 때 얼마나 오래 머무는지
    visit_days      = ('date', 'nunique'),            # 방문 일수 : 체험 기간 중 며칠이나 꾸준히 방문했는
    n_sites_visited = ('site_id', 'nunique'),         # 방문 지점 수 : 한 지점만 집중적으로 이용했는가, 여러 지점을 돌아다녔는가
    last_visit_date = ('date', 'max'),                # 마지막 방문일
).reset_index()

# 주 이용 지점(최빈값) — 여러 지점 방문 시 대표 지점 : 유저가 이용한 대표 지점
primary_site = (g_v['site_id']
                .agg(lambda s: s.mode().iloc[0])
                .rename('primary_site').reset_index())

# 첫 입실 시각(가장 이른 방문의 시각) → 시(hour)
first_hour = (g_v['first_enter_time'].min().dt.hour
              .rename('first_enter_hour').reset_index())

visit_agg = (visit_agg
             .merge(primary_site, on='user_uuid')
             .merge(first_hour,   on='user_uuid'))

# 주 지점 면적(평) 붙이기
visit_agg = (visit_agg
             .merge(site, left_on='primary_site', right_on='site_id', how='left')
             .rename(columns={'area_pyeong': 'primary_site_area'})
             .drop(columns='site_id'))

visit_agg.head()

In [ ]:
# ── 2) access_events 집계 (출입 빈도) ──
# 입실/퇴실 매칭 오류(약 28.7%)의 영향을 받지 않도록, 세션 복원 없이 '총 출입 횟수'만 집계
access_agg = (access.groupby('user_uuid').size()
              .rename('n_access_events').reset_index())

access_agg.head()

In [ ]:
# ── 3) trial_signup 기반 파생변수 (가입 시점) ──
# register = register 중복 제거본
reg_feat = register[['user_uuid', 'trial_date']].copy()
reg_feat['signup_dow']  = reg_feat['trial_date'].dt.dayofweek   # 0=월 ~ 6=일

reg_feat.head()

In [ ]:
# ── 4) 통합 테이블 생성 (register 기준 left join) ──
# data_tp = payment 중복 제거본 (target: is_payment)
master = (reg_feat
          .merge(visit_agg,  on='user_uuid', how='left')   # 방문 파생
          .merge(access_agg, on='user_uuid', how='left')   # 출입 파생
          .merge(payment[['user_uuid', 'is_payment']], on='user_uuid', how='left'))  # target

print('통합 테이블 shape:', master.shape)
print('유저 수:', master['user_uuid'].nunique())
master.head()

In [ ]:
# ── 5) 추가 파생변수 (오전/오후, 종료까지 간격, no-show) ──
master['last_visit_date'] = pd.to_datetime(master['last_visit_date'])  # ← 이 줄 추가

master['is_morning'] = (master['first_enter_hour'] < 12).astype('Int64')
master['trial_end'] = master['trial_date'].dt.normalize() + pd.Timedelta(days=2)
master['last_visit_to_end_gap'] = (master['trial_end'] - master['last_visit_date']).dt.days
master['no_show'] = master['visit_days'].isna().astype(int)

master[['is_morning', 'last_visit_to_end_gap', 'no_show']].head()

In [ ]:
# ── 6) no-show 결측 처리 ──
# 미방문 유저는 방문/출입 기반 변수가 결측 → 0으로 채움 (no_show 플래그로 별도 표시)
fill_zero = ['stay_total_sec', 'stay_mean_sec', 'visit_days',
             'n_sites_visited', 'n_access_events']
master[fill_zero] = master[fill_zero].fillna(0)

print('결측 현황(남은 것 = 미방문이라 정의상 비는 컬럼):')
print(master.isna().sum()[master.isna().sum() > 0])
print('\nno_show 유저 수:', master['no_show'].sum())
print('is_payment 결측:', master['is_payment'].isna().sum())

In [ ]:
# ── 7) 최종 피처+타겟 통합 테이블 ──
feature_cols = [
    'stay_total_sec', 'stay_mean_sec',          # 1. 체류시간
    'primary_site',                              # 2. 주 지점(범주형)
    'visit_days',                                # 3. 방문일수
    'n_access_events',                           # 4. 출입횟수
    'signup_dow',                                # 5. 가입 시점
    'n_sites_visited', 'primary_site_area',      # 6. 지점 다양성/면적
    'last_visit_to_end_gap',                     # 7. 종료까지 간격
    'is_morning',                                # 8. 오전/오후
    'no_show',                                   # 보조: 미방문 플래그
]
df_master = master[['user_uuid'] + feature_cols + ['is_payment']].copy()

print('최종 통합 테이블:', df_master.shape)
df_master.head()

In [ ]:
df_master.drop(columns=['no_show'], inplace=True)

In [ ]:
# email 컬럼 생성 (시험: 플러스 어드레싱 / 실제: 진짜 이메일로 교체)
BASE = "your_email_id"
df_master["email"] = (BASE + "+" + df_master["user_uuid"].astype(str)
                          + "_" + df_master["trial_date"].dt.strftime("%Y%m%d")
                          + "@gmail.com")

In [ ]:
print('최종 통합 테이블:', df_master.shape)
df_master.head()

### 마스터 테이블 추출 (주석 처리)

In [ ]:
# # 최종 통합 테이블을 CSV 파일로 저장
# # encoding='utf-8-sig'를 사용하면 엑셀에서 한글이 깨지지 않고 잘 열립니다.
# df_master.to_csv('df_master_output.csv', index=False, encoding='utf-8-sig')

# print("CSV 파일 추출 완료!")

<a id="sec2-8"></a>
### 2.8 [시뮬레이션용] 날짜·유저ID 포함 확장 마스터

**목적** — n8n 롤링 시뮬레이션과 월별 볼륨/전환율 점검을 위해 `user_uuid`·`trial_date`·`trial_end`를 살린 확장본을 **별도 파일**(`df_master_with_date.csv`)로 추출한다.

**설계 원칙**
- 모델 입력용 `df_master`(날짜 없음)와 `df_master_output.csv`는 **그대로 둔다** — 분류 노트북/대시보드가 이 스키마를 그대로 읽으므로 건드리면 학습 코드가 날짜를 피처로 먹거나 dtype에서 터짐.
- 확장본 = `[user_uuid, trial_date, trial_end]` + **df_master와 동일한 피처셋** + `is_payment`. 즉 meta 3개만 `drop`하면 `df_master`와 완전히 호환된다.
- `user_uuid`·`trial_date`·`trial_end`는 **피처가 아니라 키/시점**이다. 자를 때(30일 컷)와 붙일 때(예측값→유저 매핑)만 쓰이고, 모델 입력 `X`에는 절대 넣지 않는다.

In [ ]:
# ── 8) [시뮬레이션용] 날짜·유저ID 포함 확장 마스터 빌드 ──
# master 에는 user_uuid, trial_date, trial_end 가 그대로 살아있으므로 원본 재조인 불필요.
# feature_cols(7)에서 no_show 만 빼면 df_master(CODE 71 이후)와 동일 피처셋이 된다.

id_time_cols = ['user_uuid', 'trial_date', 'trial_end']       # 피처 아님 (키/시점 전용)
export_feats = [c for c in feature_cols if c != 'no_show']     # df_master 와 동일 피처셋

df_master_sim = master[id_time_cols + export_feats + ['is_payment']].copy()

# CSV 왕복 후에도 날짜형이 유지되도록 명시적으로 datetime 보장
df_master_sim['trial_date'] = pd.to_datetime(df_master_sim['trial_date'])
df_master_sim['trial_end']  = pd.to_datetime(df_master_sim['trial_end'])

print('확장 마스터 shape:', df_master_sim.shape)
print('컬럼      :', list(df_master_sim.columns))
print('기간      :', df_master_sim['trial_date'].min().date(),
      '~', df_master_sim['trial_date'].max().date())
print('유저 수   :', df_master_sim['user_uuid'].nunique())
df_master_sim.head()

In [ ]:
# ── 9) 월별 신규 유저 수 & 결제율 (우하향/드리프트 점검) ──
# "마지막 1달을 시뮬레이션 기준으로 삼는 게 합리적인가?" 를 이 표로 판단한다.
#  - 신규유저 열: 볼륨이 실제로 우하향인지, 마지막 달이 몇 명인지
#  - 결제율 열  : 전환율이 시간에 따라 드리프트했는지 (드리프트하면 앞 기간 학습→마지막 달 예측이 miscalibrated)

m = df_master_sim.copy()
m['ym'] = m['trial_date'].dt.to_period('M')   # 연-월 단위로 묶기

monthly = (m.groupby('ym')
             .agg(신규유저=('user_uuid', 'nunique'),
                  결제율  =('is_payment', 'mean'))
             .reset_index())
monthly['결제율'] = (monthly['결제율'] * 100).round(1)   # % 표기
print('[월별 볼륨 · 전환율]')
print(monthly.to_string(index=False))

# 마지막 30일 규모 = 일 배치가 하루 몇 명이나 되는지 (n8n 시뮬레이션 현실성 체크)
cutoff = m['trial_date'].max() - pd.Timedelta(days=30)
last30 = m[m['trial_date'] > cutoff]
days   = last30['trial_date'].dt.date.nunique()
print(f"\n마지막 30일: {last30['user_uuid'].nunique()}명 ({days}일치), "
      f"하루 평균 {last30['user_uuid'].nunique() / max(days, 1):.1f}명")

In [ ]:
# ── 10) 확장 마스터 CSV 추출 ──
# encoding='utf-8-sig' : 엑셀에서 한글 안 깨짐
df_master_sim.to_csv('df_master_with_date.csv', index=False, encoding='utf-8-sig')
print('df_master_with_date.csv 추출 완료:', df_master_sim.shape)

# (호환성 확인) meta 3개만 drop 하면 df_master 와 동일 피처셋인지 검증
_check = (list(df_master_sim.drop(columns=id_time_cols).columns)
          == export_feats + ['is_payment'])
print('df_master 호환(meta drop 시 동일 피처셋):', _check)

In [ ]:
#상관계수를 계산할 수치형 변수들만 선택 (user_uuid 제외)
# df_master.info() 상에서 수치형(float64, int32, int64)인 컬럼들을 자동 선택합니다.
numeric_df = df_master.select_dtypes(include=["number"])

# 3. 상관계수 행렬 계산
corr_matrix = numeric_df.corr()

# 4. 히트맵 시각화 크기 설정
plt.figure(figsize=(12, 10))

# 5. seaborn 히트맵 그리기
sns.heatmap(
    corr_matrix,
    annot=True,  # 히트맵 내부에 숫자(상관계수) 표시
    fmt=".2f",  # 소수점 둘째 자리까지 표시
    cmap="coolwarm",  # 색상 맵 (양의 상관관계는 빨강, 음의 상관관계는 파랑)
    vmin=-1,
    vmax=1,  # 상관계수 범위 -1 ~ 1 고정
    linewidths=0.5,  # 셀 사이의 간격 line 두께
    cbar_kws={"label": "상관계수"},
)

# 6. 제목 설정
plt.title("is_payment 및 주요 피처 간 상관관계 히트맵", fontsize=16, pad=20)
plt.xticks(rotation=45, ha="right")  # x축 레이블 회전으로 가독성 확보
plt.tight_layout()

# 7. 출력
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

# y축에 로그 변환을 적용하여 시각화 (0초인 유저 대응을 위해 log 스케일 적용)
sns.boxplot(
    x="is_payment", y="stay_total_sec", data=df_master, palette="coolwarm"
)

plt.yscale("log")  # 왜도가 심한 체류 시간 데이터를 위해 y축 로그 스케일 적용
plt.title(
    "결제 여부(is_payment)별 총 체류 시간(stay_total_sec) 분포",
    fontsize=14,
    pad=15,
)
plt.xlabel("결제 여부 (0: 미결제, 1: 결제)", fontsize=11)
plt.ylabel("총 체류 시간 (초, Log Scale)", fontsize=11)
plt.grid(True, axis="y", alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# x축에 방문 일수(visit_days), hue에 결제 여부(is_payment)를 설정
sns.countplot(x="visit_days", hue="is_payment", data=df_master, palette="coolwarm")

plt.title("방문 일수(visit_days)별 결제/미결제 유저 수 비교", fontsize=14, pad=15)
plt.xlabel("방문 일수 (일)", fontsize=11)
plt.ylabel("유저 수 (명)", fontsize=11)
plt.legend(title="결제 여부", labels=["미결제 (0)", "결제 (1)"])
plt.grid(True, axis="y", alpha=0.3)

plt.show()

<a id="sec3"></a>
# 3. EDA

In [ ]:
print("stay_time_second 최댓값:",
      visit['stay_time_second'].max())

print("46,651초와 정확히 같은 행 수:",
      (visit['stay_time_second'] == 46651).sum())

print("46,651초 초과 행 수:",
      (visit['stay_time_second'] > 46651).sum())

In [ ]:
# 체류시간·방문일수 EDA에 필요한 컬럼만 복사
eda_stay_visit = df_master[
    [
        'user_uuid',
        'is_payment',
        'visit_days',
        'stay_total_sec',
        'stay_mean_sec'
    ]
].copy()

# 방문일수를 정수형으로 변환
eda_stay_visit['visit_days'] = (
    eda_stay_visit['visit_days'].astype(int)
)

# 초 단위를 시간 단위로 변환
eda_stay_visit['stay_total_hour'] = (
    eda_stay_visit['stay_total_sec'] / 3600
)

In [ ]:
# 기존 stay_mean_sec는 visit_info 한 행당 평균
eda_stay_visit['stay_mean_row_hour'] = (
    eda_stay_visit['stay_mean_sec'] / 3600
)

# 실제 방문일당 평균 체류시간
# 미방문자는 0으로 나눌 수 없으므로 NaN 처리
eda_stay_visit['stay_per_day_hour'] = np.where(
    eda_stay_visit['visit_days'] > 0,
    eda_stay_visit['stay_total_hour']
    / eda_stay_visit['visit_days'],
    np.nan
)

In [ ]:
# 방문 여부 변수
eda_stay_visit['visited_flag'] = (
    eda_stay_visit['visit_days'] > 0
).astype(int)

print('EDA 데이터 크기:', eda_stay_visit.shape)
print('중복 사용자 수:', eda_stay_visit['user_uuid'].duplicated().sum())
print('\n결측치:')
print(eda_stay_visit.isna().sum())

eda_stay_visit.head()

- 전체 사용자 9,624명 / 중복 0건
- `stay_per_day_hour` 결측치 3,090명 (visit_days=0인 미방문자라서 정상)
- 미방문자는 stay_total_hour=0, visited_flag=0
- 방문자는 stay_per_day_hour가 계산됨

<a id="sec3-1"></a>
## 3.1 방문 여부별 결제율

In [ ]:
visit_flag_summary = (
    eda_stay_visit
    .groupby('visited_flag', as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),
        payment_rate=('is_payment', 'mean')
    )
)

visit_flag_summary['visited_label'] = visit_flag_summary['visited_flag'].map({
    0: '미방문',
    1: '방문'
})

visit_flag_summary['payment_rate_pct'] = (
    visit_flag_summary['payment_rate'] * 100
).round(2)

ax = visit_flag_summary.plot(
    x='visited_label',
    y='payment_rate_pct',
    kind='bar',
    legend=False,
    figsize=(6, 5)
)

plt.title('방문 여부별 결제 전환율')
plt.xlabel('방문 여부')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(20, 60)

for i, row in visit_flag_summary.iterrows():
    ax.text(
        i,
        row['payment_rate_pct'] + 2,
        f"{row['payment_rate_pct']:.1f}%\n(n={row['user_count']:,})",
        ha='center'
    )

plt.tight_layout()
plt.show()

visit_flag_summary

In [ ]:
cnt = (eda_stay_visit
       .groupby('visited_flag')['is_payment']
       .agg(결제='sum', 전체='count'))
cnt['미결제'] = cnt['전체'] - cnt['결제']
cnt.index = cnt.index.map({0: '미방문', 1: '방문'})

ax = cnt[['미결제', '결제']].plot(kind='bar', figsize=(6, 5),
                                  color=['#d9d9d9', '#4c72b0'])
plt.title('방문 여부별 결제 / 미결제 인원')
plt.xlabel('방문 여부'); plt.ylabel('사용자 수 (명)')
plt.xticks(rotation=0); plt.legend(title='결제 여부')
for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=2)
plt.tight_layout(); plt.show()

<a id="sec3-2"></a>
## 3.2 방문일수별 사용자 수 및 결제율

In [ ]:
visit_days_summary = (
    eda_stay_visit
    .groupby("visit_days", as_index=False)
    .agg(
        user_count = ("user_uuid", "count"),
        payment_count = ("is_payment", "sum"),
        payment_rate = ("is_payment", "mean")
    )
)

In [ ]:
# 결제율 퍼센트로 변환
visit_days_summary["payment_rate_pct"] = (
    visit_days_summary["payment_rate"] * 100
).round(2)

visit_days_summary

In [ ]:
# 방문일수별 결제율 막대그래프
ax = visit_days_summary.plot(
    x='visit_days',
    y='payment_rate_pct',
    kind='bar',
    legend=False,
    figsize=(8, 5)
)

plt.title('방문일수별 결제 전환율')
plt.xlabel('방문일수')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)

# 전체 평균 결제율 기준선
overall_payment_rate = (
    eda_stay_visit['is_payment'].mean() * 100
)

plt.axhline(
    overall_payment_rate,
    linestyle='--',
    label=f'전체 결제율 {overall_payment_rate:.2f}%'
)

# 막대 위에 결제율과 사용자 수 표시
for i, row in visit_days_summary.iterrows():
    ax.text(
        i,
        row['payment_rate_pct'] + 2,
        f"{row['payment_rate_pct']:.1f}%\n(n={row['user_count']:,})",
        ha='center'
    )

plt.legend()
plt.tight_layout()
plt.show()

<a id="sec3-3"></a>
## 3.3 (방문자) 결제 여부별 체류시간

In [ ]:
# 실제 방문자만 추출
visited_users = eda_stay_visit[
    eda_stay_visit["visit_days"] > 0
].copy()

# 결제 여부별 체류시간 기초통계
stay_summary = (
    visited_users
    .groupby('is_payment')
    .agg(
        user_count=('user_uuid', 'count'),

        # 체험기간 총 체류시간
        total_stay_mean=('stay_total_hour', 'mean'),
        total_stay_median=('stay_total_hour', 'median'),
        total_stay_q1=('stay_total_hour', lambda x: x.quantile(0.25)),
        total_stay_q3=('stay_total_hour', lambda x: x.quantile(0.75)),

        # 방문일당 평균 체류시간
        per_day_mean=('stay_per_day_hour', 'mean'),
        per_day_median=('stay_per_day_hour', 'median')
    )
    .reset_index()
)

# 소수점 정리
stay_summary.iloc[:, 2:] = stay_summary.iloc[:, 2:].round(2)

stay_summary

In [ ]:
# 시각화 범위 설정용 99% 분위수
stay_upper = visited_users['stay_total_hour'].quantile(0.99)

payment_0 = visited_users.loc[
    visited_users['is_payment'] == 0,
    'stay_total_hour'
]

payment_1 = visited_users.loc[
    visited_users['is_payment'] == 1,
    'stay_total_hour'
]

plt.figure(figsize=(7, 5))

plt.boxplot(
    [payment_0, payment_1],
    tick_labels=['미결제', '결제'],
    showfliers=False
)

plt.title('결제 여부별 총 체류시간 분포 — 방문자')
plt.xlabel('결제 여부')
plt.ylabel('총 체류시간 (시간)')
plt.ylim(0, stay_upper)

plt.tight_layout()
plt.show()

In [ ]:
# 같은 방문일수 내에서 결제 여부별 체류시간 비교

stay_by_visit_days = (
    visited_users
    .groupby(['visit_days', 'is_payment'], as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),

        # 체험기간 총 체류시간
        total_stay_mean=('stay_total_hour', 'mean'),
        total_stay_median=('stay_total_hour', 'median'),

        # 방문일당 평균 체류시간
        per_day_mean=('stay_per_day_hour', 'mean'),
        per_day_median=('stay_per_day_hour', 'median')
    )
)

# 소수점 둘째 자리까지 정리
stay_by_visit_days[
    [
        'total_stay_mean',
        'total_stay_median',
        'per_day_mean',
        'per_day_median'
    ]
] = (
    stay_by_visit_days[
        [
            'total_stay_mean',
            'total_stay_median',
            'per_day_mean',
            'per_day_median'
        ]
    ].round(2)
)

# 결제 여부를 보기 쉽게 표시
stay_by_visit_days['payment_label'] = (
    stay_by_visit_days['is_payment']
    .map({0: '미결제', 1: '결제'})
)

stay_by_visit_days[
    [
        'visit_days',
        'payment_label',
        'user_count',
        'total_stay_mean',
        'total_stay_median',
        'per_day_mean',
        'per_day_median'
    ]
]

In [ ]:
# 방문일수 × 결제 여부 체류시간 중앙값
per_day_pivot = (
    stay_by_visit_days
    .pivot(
        index='visit_days',
        columns='payment_label',
        values='per_day_median'
    )
)

# 보기 좋은 순서로 정렬
per_day_pivot = per_day_pivot[['미결제', '결제']]

ax = per_day_pivot.plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('방문일수별 일평균 체류시간')
plt.xlabel('방문일수')
plt.ylabel('방문일당 체류시간 중앙값 (시간)')
plt.xticks(rotation=0)

# 막대 위 숫자 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f',
        padding=3
    )

plt.legend(title='결제 여부')
plt.tight_layout()
plt.show()

In [ ]:
# 같은 방문일수 안에서 방문일당 체류시간을 3개 구간으로 나누기
stay_group_df = visited_users[
    visited_users['visit_days'].between(1, 3)
].copy()

# 방문일수별로 체류시간 순위를 구한 뒤
# 짧음 / 보통 /  3개 그룹으로 분류
stay_group_df['stay_group'] = (
    stay_group_df
    .groupby('visit_days')['stay_per_day_hour']
    .transform(
        lambda x: pd.qcut(
            x.rank(method='first'),
            q=3,
            labels = [
    '하위 1/3 (짧은 체류)',
    '중간 1/3',
    '상위 1/3 (긴 체류)'
]
        )
    )
)

# 방문일수 × 체류시간 그룹별 결제율
stay_group_summary = (
    stay_group_df
    .groupby(
        ['visit_days', 'stay_group'],
        observed=True,
        as_index=False
    )
    .agg(
        user_count=('user_uuid', 'count'),
        stay_hour_median=('stay_per_day_hour', 'median'),
        payment_count=('is_payment', 'sum'),
        payment_rate=('is_payment', 'mean')
    )
)

stay_group_summary['stay_hour_median'] = (
    stay_group_summary['stay_hour_median'].round(2)
)

stay_group_summary['payment_rate_pct'] = (
    stay_group_summary['payment_rate'] * 100
).round(2)

stay_group_summary

In [ ]:
# 방문일수 × 체류시간 그룹별 결제율 피벗
stay_payment_pivot = (
    stay_group_summary
    .pivot(
        index='visit_days',
        columns='stay_group',
        values='payment_rate_pct'
    )
)

# 그룹 순서 지정
group_order = [
   '하위 1/3 (짧은 체류)',
   '중간 1/3',
   '상위 1/3 (긴 체류)'
]

stay_payment_pivot = stay_payment_pivot[group_order]

# 그래프
ax = stay_payment_pivot.plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title('방문일수 및 방문일당 체류시간 그룹별 결제 전환율')
plt.xlabel('방문일수')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(0, 70)

# 막대 위 결제율 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3
    )

plt.legend(
    title='방문일당 체류시간 그룹'
)

plt.tight_layout()
plt.show()

<a id="sec3-4"></a>
## 3.4 주 이용 지점별 결제율

In [ ]:
site_summary = (
    df_master
    .dropna(subset=['primary_site'])
    .groupby('primary_site', as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),
        payment_rate=('is_payment', 'mean')
    )
)

site_summary['payment_rate_pct'] = (
    site_summary['payment_rate'] * 100
).round(2)

site_summary = site_summary.sort_values('payment_rate_pct', ascending=False)

ax = site_summary.plot(
    x='primary_site',
    y='payment_rate_pct',
    kind='bar',
    legend=False,
    figsize=(8, 5)
)

plt.title('주 이용 지점별 결제 전환율')
plt.xlabel('주 이용 지점')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)

for i, row in enumerate(site_summary.itertuples()):
    ax.text(
        i,
        row.payment_rate_pct + 2,
        f"{row.payment_rate_pct:.1f}%\n(n={row.user_count:,})",
        ha='center'
    )

plt.tight_layout()
plt.show()

site_summary

<a id="sec3-5"></a>
## 3.5 오전 방문자 방문 여부별 결제율

In [ ]:
morning_summary = (
    df_master
    .dropna(subset=['is_morning'])
    .groupby('is_morning', as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),
        payment_rate=('is_payment', 'mean')
    )
)

morning_summary['time_label'] = morning_summary['is_morning'].map({
    0: '오후/저녁 첫 방문',
    1: '오전 첫 방문'
})

morning_summary['payment_rate_pct'] = (
    morning_summary['payment_rate'] * 100
).round(2)

ax = morning_summary.plot(
    x='time_label',
    y='payment_rate_pct',
    kind='bar',
    legend=False,
    figsize=(7, 5)
)

plt.title('첫 방문 시간대별 결제 전환율')
plt.xlabel('첫 방문 시간대')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)

for i, row in morning_summary.iterrows():
    ax.text(
        i,
        row['payment_rate_pct'] + 2,
        f"{row['payment_rate_pct']:.1f}%\n(n={row['user_count']:,})",
        ha='center'
    )

plt.tight_layout()
plt.show()

morning_summary

<a id="sec3-6"></a>
## 3.6 출입 횟수 구간별 결제율

In [ ]:
access_df = df_master.copy()

access_df['access_group'] = pd.cut(
    access_df['n_access_events'],
    bins=[-1, 0, 2, 5, 10, 20, access_df['n_access_events'].max()],
    labels=['0회', '1~2회', '3~5회', '6~10회', '11~20회', '21회 이상']
)

access_summary = (
    access_df
    .groupby('access_group', observed=True, as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),
        payment_rate=('is_payment', 'mean')
    )
)

access_summary['payment_rate_pct'] = (
    access_summary['payment_rate'] * 100
).round(2)

ax = access_summary.plot(
    x='access_group',
    y='payment_rate_pct',
    kind='bar',
    legend=False,
    figsize=(9, 5)
)

plt.title('출입 횟수 구간별 결제 전환율')
plt.xlabel('출입 횟수 구간')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=30)
plt.ylim(0, 100)

for i, row in access_summary.iterrows():
    ax.text(
        i,
        row['payment_rate_pct'] + 2,
        f"{row['payment_rate_pct']:.1f}%\n(n={row['user_count']:,})",
        ha='center'
    )

plt.tight_layout()
plt.show()

access_summary

<a id="sec3-7"></a>
## 3.7 방문일수 × 출입 횟수

In [ ]:
visit_access_df = df_master[
    df_master['visit_days'].between(1, 3)
].copy()

visit_access_df['access_level'] = (
    visit_access_df
    .groupby('visit_days')['n_access_events']
    .transform(
        lambda x: pd.qcut(
            x.rank(method='first'),
            q=3,
            labels=['출입 적음', '출입 보통', '출입 많음']
        )
    )
)

visit_access_summary = (
    visit_access_df
    .groupby(['visit_days', 'access_level'], observed=True, as_index=False)
    .agg(
        user_count=('user_uuid', 'count'),
        access_median=('n_access_events', 'median'),
        payment_rate=('is_payment', 'mean')
    )
)

visit_access_summary['payment_rate_pct'] = (
    visit_access_summary['payment_rate'] * 100
).round(2)

visit_access_pivot = (
    visit_access_summary
    .pivot(
        index='visit_days',
        columns='access_level',
        values='payment_rate_pct'
    )
)

ax = visit_access_pivot.plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title('방문일수 및 출입횟수 그룹별 결제 전환율')
plt.xlabel('방문일수')
plt.ylabel('결제율 (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.legend(title='출입횟수 그룹')
plt.tight_layout()
plt.show()

visit_access_summary

## 3.8 방문자별 하루 평균 이용 횟수

In [ ]:
# A. 방문일 기준 — "방문한 날, 하루 평균 몇 회 이용?" (지금 코드와 동일)
sessions_per_day = visit.groupby(['user_uuid', 'date']).size()
print('[방문일 기준]')
print(f'  방문일 수            : {len(sessions_per_day):,}일')
print(f'  하루 평균 이용 횟수   : {sessions_per_day.mean():.2f}회')
print(f'  최대                 : {sessions_per_day.max()}회')

# B. 사람 기준 — "한 번이라도 방문한 사람, 1인당 하루 평균 이용 횟수?"
per_user = sessions_per_day.groupby('user_uuid').mean()   # 사람마다 '하루 평균'을 먼저 구하고
print('\n[사람 기준]')
print(f'  방문 경험 사용자 수   : {per_user.index.nunique():,}명')
print(f'  1인당 하루 평균 이용  : {per_user.mean():.2f}회')
print(f'  최대(개인 평균 중)    : {per_user.max():.2f}회')

In [ ]:
import matplotlib.pyplot as plt

v = visit.sort_values(['user_uuid', 'date', 'first_enter_time']).copy()
v['next_enter'] = v.groupby(['user_uuid', 'date'])['first_enter_time'].shift(-1)

# 같은 날 다음 세션이 있는 행 = 중간 외출 케이스
brk = v.dropna(subset=['next_enter']).copy()
brk['leave_hour']  = brk['last_leave_time'].dt.hour
brk['return_hour'] = brk['next_enter'].dt.hour
brk['gap_min']     = (brk['next_enter'] - brk['last_leave_time']).dt.total_seconds() / 60

print(f'중간 외출(같은 날 재입실) 건수 : {len(brk):,}')
print(f'외출 시간 중앙값               : {brk["gap_min"].median():.0f}분')
print(f'나간 시각 상위 3개 :\n{brk["leave_hour"].value_counts().head(3)}')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
brk['leave_hour'].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=ax[0], color='#c44e52')
ax[0].set_title('나간 시각(이전 세션 퇴실)'); ax[0].set_xlabel('시'); ax[0].set_ylabel('건수')
brk['return_hour'].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=ax[1], color='#4c72b0')
ax[1].set_title('돌아온 시각(다음 세션 입실)'); ax[1].set_xlabel('시')
for a in ax:
    a.axvspan(11.5, 13.5, color='orange', alpha=.15)   # 점심대(12~13시)
    a.axvspan(17.5, 19.5, color='green',  alpha=.10)   # 저녁대(18~19시)
plt.tight_layout(); plt.show()

<a id="sec4"></a>
# 4. 결론

### 1. 방문일수는 결제 전환과 가장 뚜렷한 관계를 보였다.

    방문일수가 증가할수록 유료 결제 전환율도 함께 증가하는 경향이 확인되었다.
    특히 2일 이상 방문한 이용자부터 평균 결제율을 상회하였으며, 반복 방문이 결제 전환과 밀접한 관련이 있는 것으로 나타났다.

### 2. 체류시간은 예상과 다른 결과를 보였다.

    초기 가설은 "오래 체류할수록 결제 가능성이 높다"였으나,
    실제 분석에서는 미결제 이용자의 평균 체류시간이 결제 이용자보다 더 길게 나타났다.
    따라서 체류시간만으로는 결제 여부를 설명하기 어려웠다.

### 3. 출입횟수는 결제와 큰 관련성을 보이지 않았다.

    출입횟수 구간별 결제율은 큰 차이가 나타나지 않았다.
    출입 빈도 자체보다 방문일수와 같은 반복 이용 패턴이 더 중요한 변수일 가능성이 있다.

### 4. 방문 습관이 체류시간보다 중요한 행동 지표일 가능성이 있다.

    동일한 무료체험 기간 동안 한 번 오래 이용하는 것보다 여러 날 반복해서 방문한 이용자가 결제 전환율이 높았다.
    공유오피스 서비스에서는 '이용 시간'보다 '반복 방문 경험'이 결제 전환과 더 밀접한 관계를 보이는 것으로 판단된다.
